<a href="https://colab.research.google.com/github/2177757arturo/PIA_/blob/main/IA_VISS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import serial
import time
import cv2
import numpy as np
import pickle

# <-- ¡Asegúrate de cambiar esto por el puerto COM exacto de tu ESP32!
PUERTO = 'COM7'
BAUDIOS = 115200

# --- 1. CARGAR EL MODELO DE MACHINE LEARNING ---
try:
    with open('modelo_colores.pkl', 'rb') as archivo:
        modelo_ia = pickle.load(archivo)
    print(" : Cerebro de Machine Learning cargado con éxito.")
except FileNotFoundError:
    print("❌ Error: No se encontró 'modelo_colores.pkl'. Corre tu programa de entrenamiento primero.")
    exit()

# --- 2. CONEXIÓN SERIAL SEGURA ---
try:
    esp32 = serial.Serial(PUERTO, BAUDIOS, timeout=0.1)
    esp32.setDTR(False)
    esp32.setRTS(False)
    time.sleep(2)
    print(": Conexión con ESP32 establecida.")
except Exception as e:
    print(f"❌ Error crítico de conexión: {e}")
    exit()

def predecir_con_IA(imagen_recortada):
    # Extraemos las mismas características que usamos en el entrenamiento
    hsv = cv2.cvtColor(imagen_recortada, cv2.COLOR_BGR2HSV)
    promedio_h = np.mean(hsv[:,:,0])
    promedio_s = np.mean(hsv[:,:,1])
    promedio_v = np.mean(hsv[:,:,2])

    datos_pieza = np.array([[promedio_h, promedio_s, promedio_v]])

    # La Inteligencia Artificial hace la predicción matemática
    prediccion = modelo_ia.predict(datos_pieza)
    return prediccion[0] # <-- ¡Corregido al índice 0!

print("💤 Sistema ML en reposo. Cámara APAGADA. Esperando al sensor de la banda...")

# --- 3. BUCLE PRINCIPAL (Mecatrónica + ML) ---
while True:
    try:
        # LECTURA SERIAL BLINDADA
        if esp32.in_waiting > 0:
            mensaje = esp32.readline().decode('utf-8', errors='ignore').strip()

            # FILTRO ANTI-FALSOS: Exigimos que el mensaje sea EXACTAMENTE igual, sin basura
            if mensaje == "TOMAR_FOTO":
                print("\n📸 ¡Sensor activado! Encendiendo cámara y analizando con ML...")

                # RECUERDA: Cambia a 0, 1 o 2 dependiendo de qué número te funcionó en las pruebas
                cap = cv2.VideoCapture(1)

                for _ in range(30): # Calentamiento de lente para evitar fotos oscuras
                    cap.read()

                ret, frame = cap.read()

                if ret:
                    frame = cv2.flip(frame, 1) # Efecto espejo opcional

                    # --- AJUSTA ESTAS COORDENADAS A TU CUADRITO BLANCO ---
                    y1, y2, x1, x2 = 200, 280, 280, 360

                    cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 255, 255), 2)
                    cv2.imshow("IA de WARRY - Vision ML", frame)
                    cv2.waitKey(1000)

                    zona_analisis = frame[y1:y2, x1:x2]

                    # Usamos el modelo entrenado para decidir
                    color_detectado = predecir_con_IA(zona_analisis)

                    if color_detectado == 'R':
                        print("-> 🔴 Predicción ML: ROJO. Mandando orden al ESP32...")
                    elif color_detectado == 'V':
                        print("-> 🟢 Predicción ML: VERDE. Mandando orden al ESP32...")
                    else:
                        print("-> 🔵 Predicción ML: AZUL. Mandando orden al ESP32...")

                    # Enviar la letra al ESP32
                    esp32.write(color_detectado.encode('utf-8'))

                cap.release()
                cv2.destroyAllWindows()

                # FILTRO ANTI-FALSOS: Limpiamos la tubería de comunicación por si hubo rebote en el sensor
                esp32.reset_input_buffer()

                print("💤 Cámara apagada. Sistema en reposo esperando nueva pieza...")

        time.sleep(0.05)

    except KeyboardInterrupt:
        print("\n⏹️ Apagando sistema de Inteligencia Artificial...")
        break

esp32.close()

ModuleNotFoundError: No module named 'serial'